# WISE-Only AGN Variability Manifold + Zeltyn et al. (2024) Projection

This notebook:
1. Rebuilds the WISE-only manifold from **Sample A** (Hemmati et al. 2026, arXiv:2601.08037)
2. Trains a UMAP embedding using GP-interpolated W1 light curves with DTW distance
3. Loads the **Zeltyn et al. (2024)** CL-AGN sample (arXiv:2401.01933, SDSS-V first-year)
4. Projects the Zeltyn targets onto the trained manifold using `mapper.transform()`
5. Visualises whether the new CL-AGNs land near the known turn-on/turn-off region

**Reference configuration from the paper:**
- GP regression (`unify_lc_gp`), RBF kernel `length_scale=200`, `xres=160`
- Single band: **W1** (clearest manifold per Figure 4–7 of the paper)
- Normalise by σ-clipped max of W1
- UMAP: `n_neighbors=50`, `min_dist=0.9`, `metric=dtw_distance`

## 0. Setup

In [1]:
import sys, os

# The notebook lives at CLAGN/wise_manifold_zeltyn.ipynb
# code_src and data are sibling folders: CLAGN/code_src/ and CLAGN/data/
NOTEBOOK_DIR = os.path.abspath('')   # cwd when running a notebook = the notebook's folder
sys.path.insert(0, os.path.join(NOTEBOOK_DIR, 'code_src'))

import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import astropy.units as u
import umap
from astropy.table import Table
from astropy.coordinates import SkyCoord
from astroquery.vizier import Vizier
from mpl_toolkits.axes_grid1 import make_axes_locatable

from data_structures import MultiIndexDFObject
from wise_functions import wise_get_lightcurves
from sample_selection import (
    get_sdss_sample,
    get_lamassa_sample,
    get_macleod16_sample,
    get_ruan_sample,
    get_macleod19_sample,
    get_yang_sample,
    get_sheng_sample,
    get_green_sample,
    get_lopeznavas_sample,
    get_hon_sample,
    get_graham_sample,
    clean_sample,
)
from AGNzoo_functions import (
    unify_lc_gp,
    stat_bands,
    combine_bands,
    normalize_clipmax_objects,
    dtw_distance,
    shuffle_datalabel,
    stretch_small_values_arctan,
    translate_bitwise_sum_to_labels,
)

# Reproducibility
RANDOM_STATE = 20
DATA_DIR = os.path.join(NOTEBOOK_DIR, 'data')
os.makedirs(DATA_DIR, exist_ok=True)
print('code_src path:', os.path.join(NOTEBOOK_DIR, 'code_src'))
print('data path:    ', DATA_DIR)

code_src path: /Users/shemmati/Desktop/CLAGN/code_src
data path:     /Users/shemmati/Desktop/CLAGN/data


## 1. Load Sample A from pre-built parquet

The full Sample A light curve file (`df_lc_020724.parquet.gzip`) contains all 2,078 objects
from Hemmati et al. 2026 with multi-band photometry including WISE W1/W2.
Labels are bitwise integers encoding class membership.

In [ ]:
# Bit-to-label mapping (from Hemmati+2026 Table 1)
BITS = {
    1:    'SDSS_QSO',
    2:    'WISE_Variable',
    4:    'Optical_Variable',
    8:    'Galex_Variable',
    16:   'CLAGN',       # Turn-on
    32:   'CLAGN',       # Turn-off
    64:   'SPIDER_BL',
    128:  'SPIDER_AGNBL',
    256:  'SPIDER_QSOBL',
    2048: 'TDE',
}

def decode_label(bitsum):
    """Return the most specific human-readable label for a bitwise integer."""
    n = int(bitsum)
    # CLAGN takes highest priority (Turn-on OR Turn-off)
    if n & 16 or n & 32:
        return 'CLAGN'
    if n & 2048:
        return 'TDE'
    priority = ['WISE_Variable', 'Optical_Variable', 'Galex_Variable',
                'SPIDER_BL', 'SPIDER_AGNBL', 'SPIDER_QSOBL', 'SDSS_QSO']
    for p in priority:
        bit = next(k for k, v in BITS.items() if v == p)
        if n & bit:
            return p
    return str(bitsum)

# ── Load pre-built parquet ──────────────────────────────────────────────────
PREBUILT = os.path.join(DATA_DIR, 'df_lc_020724.parquet.gzip')
df_lc_A = MultiIndexDFObject(data=pd.read_parquet(PREBUILT))

# Remove label '64' (SPIDER-only objects — too large a sub-class, per original notebook)
df_lc_A.data = df_lc_A.data[df_lc_A.data.index.get_level_values('label') != '64']

# Keep only W1 band for the WISE-only manifold
df_lc_A.data = df_lc_A.data[df_lc_A.data.index.get_level_values('band') == 'W1']

n_obj = df_lc_A.data.index.get_level_values('objectid').nunique()
print(f'Sample A objects with W1 data: {n_obj}')
print('Bands:', df_lc_A.data.index.get_level_values('band').unique().tolist())

---
## 3. Preprocess Sample A – GP Interpolation

Using **W1 only** (clearest manifold structure per Figure 4–7 of the paper).
GP with `RBF(length_scale=200)` interpolates each light curve onto
`x_wise = linspace(0, 4000, xres=160)`.

In [ ]:
BANDS = ['W1']   # W1 only; change to ['W1', 'W2'] to include W2
XRES  = 160      # grid resolution (matches paper)

# GP interpolation (numplots=0 suppresses per-object plots; set to 3 to inspect a few)
objects_A, dobjects_A, flabels_A, keeps_A = unify_lc_gp(
    df_lc_A.data, BANDS, xres=XRES, numplots=0, low_limit_size=5
)

print(f'Objects surviving GP cut: {len(objects_A)}')

In [ ]:
# Basic statistics – fractional variation and mean brightness per band
fvar_A, maxarray_A, meanarray_A = stat_bands(objects_A, dobjects_A, BANDS, sigmacl=5)

# Concatenate bands into one long vector per object
dat_notnormal_A = combine_bands(objects_A, BANDS)

# Normalise: divide all values by the σ-clipped max of band 0 (W1)
dat_A = normalize_clipmax_objects(dat_notnormal_A, maxarray_A, band=0)

# Drop any objects whose feature vector contains NaN (e.g. max=0 after clipping)
valid_mask = ~np.isnan(dat_A).any(axis=1)
n_dropped = (~valid_mask).sum()
if n_dropped:
    print(f'Dropping {n_dropped} objects with NaN features (bad normalisation)')
dat_A      = dat_A[valid_mask]
flabels_A  = [l for l, v in zip(flabels_A, valid_mask) if v]
fvar_A     = fvar_A[:, valid_mask]
maxarray_A = maxarray_A[:, valid_mask]
meanarray_A = meanarray_A[:, valid_mask]

# Shuffle for UMAP (avoids ordering bias)
data_A, fzr_A, perm_A = shuffle_datalabel(dat_A, flabels_A)
fvar_arr_A   = fvar_A[:, perm_A]
maxarray_pA  = maxarray_A[:, perm_A]
meanarray_pA = meanarray_A[:, perm_A]

print(f'Feature matrix shape: {data_A.shape}')  # (n_objects, XRES * n_bands)

### 3.1  Build label index for plotting

The labels here are plain strings (e.g. `'SDSS'`, `'LaMassa 15'`).
We build a `labc` dict mapping each label to the list of object indices
that carry it.

In [ ]:
labc_A = {}
for idx, lab in enumerate(fzr_A):
    human = decode_label(lab)
    labc_A.setdefault(human, []).append(idx)

print('Classes in Sample A manifold:')
for k, v in sorted(labc_A.items(), key=lambda x: -len(x[1])):
    print(f'  {k:25s}: {len(v)} objects')

---
## 4. Train UMAP Manifold on Sample A

DTW distance gives the cleanest separation (per the paper).  
⚠️ DTW is expensive – for ~200 objects it takes a few minutes; for 1000+ it can take 30+ min.
A fast alternative (`metric='manhattan'`) is included as a fallback.

In [ ]:
USE_DTW = True   # set False to use manhattan (much faster)

metric = dtw_distance if USE_DTW else 'manhattan'
metric_name = 'DTW' if USE_DTW else 'Manhattan'

print(f'Training UMAP with {metric_name} distance on {data_A.shape[0]} objects …')
mapper = umap.UMAP(
    n_neighbors=50,
    min_dist=0.9,
    metric=metric,
    random_state=RANDOM_STATE,
).fit(data_A)

print('Training complete.')
print(f'Embedding shape: {mapper.embedding_.shape}')

### 4.1  Visualise Sample A manifold

In [ ]:
labc_A = {}
for idx, lab in enumerate(fzr_A):
    human = decode_label(lab)
    labc_A.setdefault(human, []).append(idx)

print('Classes in Sample A manifold:')
for k, v in sorted(labc_A.items(), key=lambda x: -len(x[1])):
    print(f'  {k:25s}: {len(v)} objects')

---
## 5. Build Zeltyn et al. (2024) Sample

Zeltyn et al. (2024, arXiv:2401.01933) present 113 spectroscopically confirmed
CL-AGNs from SDSS-V BHM first-year data.  We try to retrieve the table from
VizieR automatically; if unavailable, a CSV fallback is provided.

In [ ]:
def sdss_jname_to_skycoord(jname: str) -> SkyCoord | None:
    """
    Parse an SDSS J-name (e.g. 'SDSS J002806.86-052332.8' or 'J002806.86-052332.8') to SkyCoord.
    """
    # Strip optional 'SDSS ' prefix and the mandatory 'J'
    j = re.sub(r'^(SDSS\s*)?J', '', jname.strip())
    m = re.match(r'(\d{2})(\d{2})(\d{2}\.\d+)([+-])(\d{2})(\d{2})(\d{2}\.\d+)$', j)
    if m is None:
        warnings.warn(f'Could not parse J-name: {jname}')
        return None
    ra_str  = f"{m.group(1)}:{m.group(2)}:{m.group(3)}"
    dec_str = f"{m.group(4)}{m.group(5)}:{m.group(6)}:{m.group(7)}"
    return SkyCoord(ra_str, dec_str, unit=(u.hourangle, u.deg))


def build_zeltyn_sample_table(zeltyn_df: pd.DataFrame) -> Table:
    """
    Convert a DataFrame with columns ['Name', 'Class', 'z'] (and optionally 'RA'/'Dec')
    into an astropy sample_table compatible with wise_get_lightcurves().

    'Class' should be 'CL-AGN' or 'EVQ' (we keep only CL-AGNs by default).
    """
    # Keep confirmed CL-AGNs (both core and RM subsets)
    cl = zeltyn_df[zeltyn_df['Class'].str.startswith('CL-AGN')].copy()

    coords_z, labels_z = [], []
    for _, row in cl.iterrows():
        # Prefer explicit RA/Dec columns if present
        if 'RA' in row and 'Dec' in row and pd.notna(row['RA']):
            c = SkyCoord(ra=row['RA'] * u.deg, dec=row['Dec'] * u.deg)
        else:
            c = sdss_jname_to_skycoord(row['Name'])
        if c is not None:
            coords_z.append(c)
            labels_z.append('Zeltyn24')

    tbl = clean_sample(coords_z, labels_z, consolidate_nearby_objects=False)
    print(f'Zeltyn CL-AGN sample size: {len(tbl)}')
    return tbl

In [ ]:
# ── Try VizieR first ──────────────────────────────────────────────────────────
# Zeltyn+2024, ApJS 270, 27  →  VizieR catalog  J/ApJS/270/27
# If the catalog ID has changed, try: Vizier.find_catalogs('2024ApJS..270...27Z')

zeltyn_df = None
try:
    Vizier.ROW_LIMIT = -1
    catalog_list = Vizier.find_catalogs('2024ApJS..270...27Z')
    if catalog_list:
        cats = Vizier.get_catalogs(list(catalog_list.keys()))
        # Table 5 contains the CL-AGN list; inspect cats[i].meta to identify it
        for i, cat in enumerate(cats):
            print(f'Table {i}: {cat.meta.get("name", "?")} – {len(cat)} rows, cols: {cat.colnames[:8]}')
        # Adjust the index below to select the CL-AGN table (usually the longest one)
        t5 = cats[0].to_pandas()
        # Rename columns to expected names (adjust if VizieR column names differ)
        # Common VizieR names: 'Name' or 'SDSS', 'Class' or 'Cl', 'z' or 'zsp'
        t5 = t5.rename(columns={'SDSS': 'Name', 'Cl': 'Class', 'zsp': 'z'})
        zeltyn_df = t5
        print(f'Downloaded {len(zeltyn_df)} rows from VizieR.')
    else:
        print('VizieR catalog not found – will use CSV fallback.')
except Exception as e:
    print(f'VizieR query failed ({e}) – will use CSV fallback.')

In [ ]:
# ── CSV fallback ──────────────────────────────────────────────────────────────
# If VizieR is unavailable, create a CSV file at the path below with columns:
#   Name        – SDSS J-name  (e.g.  SDSS J002806.86-052332.8)
#   Class       – 'CL-AGN' or 'EVQ'
#   z           – spectroscopic redshift
# Optionally add columns RA and Dec (decimal degrees) to skip J-name parsing.

ZELTYN_CSV = os.path.join(DATA_DIR, 'zeltyn2024_table5.csv')

if zeltyn_df is None:
    if os.path.exists(ZELTYN_CSV):
        zeltyn_df = pd.read_csv(ZELTYN_CSV)
        print(f'Loaded {len(zeltyn_df)} rows from {ZELTYN_CSV}')
    else:
        raise FileNotFoundError(
            f'Neither VizieR nor the CSV fallback ({ZELTYN_CSV}) is available.\n'
            'Please download Table 5 from https://arxiv.org/abs/2401.01933 '
            'and save it as a CSV with columns: Name, Class, z.'
        )

# Inspect
print(zeltyn_df.head())
print('Class counts:', zeltyn_df['Class'].value_counts().to_dict())

In [ ]:
# Build the sample_table for Zeltyn CL-AGNs
sample_table_Z = build_zeltyn_sample_table(zeltyn_df)

---
## 6. Fetch WISE Light Curves for Zeltyn Targets

In [ ]:
ZELTYN_PARQUET = os.path.join(DATA_DIR, 'df_lc_zeltyn_wise.parquet.gzip')

if os.path.exists(ZELTYN_PARQUET):
    print('Loading cached Zeltyn light curves …')
    df_lc_Z = MultiIndexDFObject(data=pd.read_parquet(ZELTYN_PARQUET))
else:
    print('Fetching WISE light curves for Zeltyn targets …')
    df_lc_Z = wise_get_lightcurves(
        sample_table_Z,
        radius=1.0,
        bandlist=['WISE_W1', 'WISE_W2'],
    )
    df_lc_Z.data.to_parquet(ZELTYN_PARQUET, compression='gzip')
    print(f'Saved to {ZELTYN_PARQUET}')

# Rename bands (same as Section 2.1)
df_lc_Z.data.index = df_lc_Z.data.index.set_levels(
    df_lc_Z.data.index.levels[df_lc_Z.data.index.names.index('band')].str.replace('WISE_', '', regex=False),
    level='band'
)

n_with_data = df_lc_Z.data.index.get_level_values('objectid').nunique()
print(f'Zeltyn objects with WISE data: {n_with_data} / {len(sample_table_Z)}')

---
## 7. Preprocess Zeltyn Light Curves

**Identical** pipeline to Section 3 so the feature vectors are comparable.

In [ ]:
objects_Z, dobjects_Z, flabels_Z, keeps_Z = unify_lc_gp(
    df_lc_Z.data, BANDS, xres=XRES, numplots=0, low_limit_size=5
)
print(f'Zeltyn objects surviving GP cut: {len(objects_Z)}')

fvar_Z, maxarray_Z, meanarray_Z = stat_bands(objects_Z, dobjects_Z, BANDS, sigmacl=5)
dat_notnormal_Z = combine_bands(objects_Z, BANDS)

# Normalise using SAME normalisation band (W1, index 0)
dat_Z = normalize_clipmax_objects(dat_notnormal_Z, maxarray_Z, band=0)

print(f'Zeltyn feature matrix shape: {dat_Z.shape}')

---
## 8. Project Zeltyn Targets onto the Trained Manifold

`mapper.transform()` performs **out-of-sample** embedding — no retraining.

In [ ]:
embedding_Z = mapper.transform(dat_Z)
print(f'Zeltyn embedding shape: {embedding_Z.shape}')

---
## 9. Final Plot – Zeltyn CL-AGNs on the Sample A Manifold

In [ ]:
# ── Identify CLAGN and SDSS indices using decoded labels ─────────────────────
clagn_idx_A = labc_A.get('CLAGN', [])
sdss_idx_A  = labc_A.get('SDSS_QSO', [])

print(f'CLAGNs in manifold: {len(clagn_idx_A)}')
print(f'SDSS QSOs in manifold: {len(sdss_idx_A)}')

fig, ax = plt.subplots(figsize=(9, 8))

# Background: SDSS QSOs (grey)
ax.scatter(
    mapper.embedding_[sdss_idx_A, 0], mapper.embedding_[sdss_idx_A, 1],
    s=40, c='lightgrey', edgecolors='grey', linewidths=0.3,
    alpha=0.7, label=f'SDSS QSO (n={len(sdss_idx_A)})', zorder=1,
)

# Known CLAGNs from Sample A (blue)
ax.scatter(
    mapper.embedding_[clagn_idx_A, 0], mapper.embedding_[clagn_idx_A, 1],
    s=80, c='steelblue', edgecolors='navy', linewidths=0.5,
    alpha=0.85, label=f'Known CLAGNs – Sample A (n={len(clagn_idx_A)})', zorder=2,
)

# Zeltyn+2024 CL-AGNs (red stars)
ax.scatter(
    embedding_Z[:, 0], embedding_Z[:, 1],
    s=150, c='crimson', marker='*', edgecolors='darkred', linewidths=0.5,
    alpha=0.9, label=f'Zeltyn+2024 CL-AGNs (projected, n={len(embedding_Z)})', zorder=3,
)

ax.set_title(
    f'WISE W1 UMAP  ({metric_name} distance)\n'
    r'Sample A manifold + Zeltyn et al. (2024) projection',
    size=13,
)
ax.legend(fontsize=10, loc='best', framealpha=0.8)
ax.axis('off')

plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'zeltyn_projection.pdf'), bbox_inches='tight', dpi=150)
plt.show()
print('Saved to', os.path.join(DATA_DIR, 'zeltyn_projection.pdf'))

### 9.1  Per-class 2D density comparison

In [ ]:
from scipy.stats import gaussian_kde

# Compute KDE for known CLAGNs and Zeltyn targets on the same grid
x_all = np.concatenate([mapper.embedding_[:, 0], embedding_Z[:, 0]])
y_all = np.concatenate([mapper.embedding_[:, 1], embedding_Z[:, 1]])
x_grid = np.linspace(x_all.min() - 0.5, x_all.max() + 0.5, 200)
y_grid = np.linspace(y_all.min() - 0.5, y_all.max() + 0.5, 200)
XX, YY = np.meshgrid(x_grid, y_grid)
positions = np.vstack([XX.ravel(), YY.ravel()])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (pts, title, cmap) in zip(axes, [
    (mapper.embedding_[clagn_idx_A], 'Known CLAGNs\n(Sample A)', 'Blues'),
    (embedding_Z,                    'Zeltyn+2024 CL-AGNs\n(projected)', 'Reds'),
]):
    if len(pts) > 2:
        kde = gaussian_kde(pts.T)
        Z_kde = kde(positions).reshape(XX.shape)
        ax.contourf(XX, YY, Z_kde, levels=12, cmap=cmap, alpha=0.7)
        ax.contour(XX, YY, Z_kde, levels=5, colors='k', linewidths=0.4, alpha=0.4)
    # Scatter all Sample A as faint backdrop
    ax.scatter(mapper.embedding_[:, 0], mapper.embedding_[:, 1],
               s=10, c='lightgrey', alpha=0.4, zorder=0)
    ax.scatter(pts[:, 0], pts[:, 1], s=30, alpha=0.8, zorder=2)
    ax.set_title(title, size=12)
    ax.axis('off')

plt.suptitle('Density comparison on WISE W1 manifold', size=14)
plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'zeltyn_kde_comparison.pdf'), bbox_inches='tight', dpi=150)
plt.show()

### 9.2  Summary statistics – overlap with known CLAGN region

In [ ]:
from scipy.spatial import ConvexHull
from matplotlib.patches import Polygon as MplPolygon
from matplotlib.collections import PatchCollection

# Compute the convex hull of known CLAGN region in the manifold
clagn_pts = mapper.embedding_[clagn_idx_A]

if len(clagn_pts) >= 3:
    hull = ConvexHull(clagn_pts)
    hull_path = clagn_pts[hull.vertices]

    from matplotlib.path import Path
    hull_mpl_path = Path(np.vstack([hull_path, hull_path[0]]))

    # Count Zeltyn targets inside the hull
    inside = hull_mpl_path.contains_points(embedding_Z)
    n_inside = inside.sum()
    frac_inside = n_inside / len(embedding_Z) if len(embedding_Z) > 0 else 0

    print(f'Zeltyn CL-AGNs inside the known-CLAGN convex hull:')
    print(f'  {n_inside} / {len(embedding_Z)}  ({100*frac_inside:.1f}%)')

    # Visualise
    fig, ax = plt.subplots(figsize=(8, 7))
    ax.scatter(mapper.embedding_[:, 0], mapper.embedding_[:, 1],
               s=20, c='lightgrey', alpha=0.5, zorder=0)
    ax.scatter(clagn_pts[:, 0], clagn_pts[:, 1],
               s=60, c='steelblue', edgecolors='navy', linewidths=0.4,
               alpha=0.8, label='Known CLAGNs', zorder=2)

    poly = MplPolygon(hull_path, closed=True,
                      facecolor='steelblue', edgecolor='navy',
                      alpha=0.15, linewidth=1.2, linestyle='--', zorder=1)
    ax.add_patch(poly)

    colors_Z = ['crimson' if ins else 'orange' for ins in inside]
    ax.scatter(embedding_Z[:, 0], embedding_Z[:, 1],
               s=120, c=colors_Z, marker='*', edgecolors='darkred',
               linewidths=0.4, alpha=0.9,
               label=f'Zeltyn+2024 (red=inside hull, {n_inside}/{len(embedding_Z)})', zorder=3)

    ax.set_title('Zeltyn+2024 targets vs. known CLAGN region', size=13)
    ax.legend(fontsize=9, loc='best')
    ax.axis('off')
    plt.tight_layout()
    plt.savefig(os.path.join(DATA_DIR, 'zeltyn_hull_overlap.pdf'), bbox_inches='tight', dpi=150)
    plt.show()
else:
    print('Not enough known CLAGN points to form a convex hull.')

---
## 10. Save Embeddings

Save the 2-D coordinates for use in the proposal figures.

In [ ]:
# Sample A embedding
emb_A_df = pd.DataFrame({
    'umap_x': mapper.embedding_[:, 0],
    'umap_y': mapper.embedding_[:, 1],
    'label':  fzr_A,
})
emb_A_df.to_csv(os.path.join(DATA_DIR, 'sampleA_embedding.csv'), index=False)

# Zeltyn embedding
emb_Z_df = pd.DataFrame({
    'umap_x': embedding_Z[:, 0],
    'umap_y': embedding_Z[:, 1],
    'label':  flabels_Z,
})
emb_Z_df.to_csv(os.path.join(DATA_DIR, 'zeltyn_embedding.csv'), index=False)

print('Embeddings saved to', DATA_DIR)

---
## Appendix – Notes

### Band naming
- `wise_get_lightcurves()` returns bands named `'WISE_W1'` / `'WISE_W2'`.
- `unify_lc_gp()` internally checks `band == 'W1'` to apply the WISE GP kernel
  (`RBF(length_scale=200)`) and time grid (`linspace(0, 4000, xres)`).
- The renaming in Section 2.1 / 6 bridges this gap.

### Reproducibility
- Set `RANDOM_STATE` at the top; UMAP with DTW is deterministic given the same seed.
- The parquet caches in `../data/` prevent redundant S3 downloads.

### Extending to W2
- Change `BANDS = ['W1']` to `BANDS = ['W1', 'W2']`.
- The paper (Hemmati+2026) reports W1-only gives the cleanest CLAGN separation,
  but W1+W2 carries more information for the full population.

### Zeltyn table
- If VizieR lookup fails, download Table 5 from the published paper
  (arXiv:2401.01933) and save as `../data/zeltyn2024_table5.csv` with columns
  `Name`, `Class`, `z`.  Optionally add `RA` and `Dec` (decimal degrees) to
  skip J-name coordinate parsing.